# Domain Specialization — Aggregate Routing Statistics + UMAP

Generates `domain_specialization.json` (aggregate per-domain routing statistics:
`activation_rate`, `avg_prob`, `specialization_score`, `layer_divergence`, `domain_rate`,
`top_specialists`) plus `domain_specialization_umap.json` (2D UMAP projection of the same
per-(layer, expert) activation-rate vectors, one dimension per domain).

6 domains (`code`, `math`, `biomedical`, `legal`, `creative_writing`, `conversational`), one
long (~300-400 word) passage each, all rewritten from scratch for stylistic consistency.
No dedicated "baseline" passage: since none of these 6 domains is meant to be neutral/generic
text, `specialization_score` and `layer_divergence` are computed against a **synthetic
baseline** — the mean activation rate across the 6 domains themselves, per (layer, expert) —
instead of a 7th hand-authored passage.

Run on a Colab A100 GPU runtime.


In [ ]:
import importlib.util
import subprocess
import sys


def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


if importlib.util.find_spec("torch") is None:
    pip_install("torch")

pip_install("transformers>=5.0.0,<6.0.0", "accelerate", "umap-learn", "numpy", "scikit-learn")

print("Dependency installation complete.")


In [ ]:
import json
import os
from collections import defaultdict

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "allenai/OLMoE-1B-7B-0924"
OUT_PATH = "domain_specialization.json"
UMAP_OUT_PATH = "domain_specialization_umap.json"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# No attn_implementation="eager" needed here -- this script never extracts attention
# weights, only router_logits, so the default (faster) sdpa attention is fine.
model, loading_info = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
    output_loading_info=True,
)
model.eval()

assert not loading_info["missing_keys"], (
    f"Some model weights were NOT loaded from the checkpoint (randomly initialized "
    f"instead): {loading_info['missing_keys']}"
)
print(f"unexpected_keys (informational): {loading_info.get('unexpected_keys', [])}")

config = model.config
assert config.num_hidden_layers == 16, f"Expected 16 layers, got {config.num_hidden_layers}"
assert config.num_experts == 64, f"Expected 64 experts, got {config.num_experts}"
assert config.num_experts_per_tok == 8, f"Expected top-8 routing, got {config.num_experts_per_tok}"

num_experts = config.num_experts
top_k_experts = config.num_experts_per_tok
num_layers = config.num_hidden_layers

print(f"Loaded {MODEL_ID}: {num_layers} layers, {num_experts} experts, top-{top_k_experts} routing")


In [ ]:
# 6 domains x 1 long, coherent, grammatical passage each. All 6 passages were written from
# scratch for stylistic consistency (length, structure) -- including code/legal/biomedical,
# not just the 2 domains that are entirely new (math, conversational). The old "poetry"
# domain is retired in favor of "creative_writing". One prompt per domain keeps this to 6
# forward passes total; loading the model once is the fixed cost, prompt length barely
# affects memory, so longer passages here capture much richer per-domain routing signal.
DOMAIN_PROMPTS = {
    "code": [
        "Most production codebases begin with an interface contract before a single line of "
        "business logic gets written. A REST API endpoint is typically documented first with "
        "its request shape, response shape, and error codes, so that frontend and backend "
        "teams can work in parallel against a shared expectation rather than waiting on each "
        "other. Once the contract is settled, the backend team implements a service layer "
        "that validates input, applies business rules, and delegates persistence to a "
        "repository layer, keeping raw database queries out of the request handlers "
        "entirely.\n\n"
        "Consider a function that searches a sorted array for a target value using binary "
        "search. The function compares the target against the middle element, and if they "
        "do not match, discards the half of the array that cannot contain the target, "
        "repeating the process on the remaining half. This halves the search space on every "
        "comparison, giving binary search a logarithmic time complexity of O(log n), a "
        "dramatic improvement over the O(n) cost of scanning the array element by element, "
        "especially once the array grows into the millions of entries.\n\n"
        "Concurrency introduces its own category of bugs that rarely show up in "
        "single-threaded testing. A race condition occurs when two threads read and write "
        "shared state without proper synchronization, producing a result that depends on "
        "unpredictable timing rather than program logic. Developers guard against this with "
        "locks, atomic operations, or by redesigning the system around immutable data "
        "structures and message passing, so that no two threads ever mutate the same memory "
        "at the same time.\n\n"
        "Once a feature is implemented, it still has to survive code review and continuous "
        "integration before merging. A pull request typically triggers an automated pipeline "
        "that runs the unit test suite, checks code coverage, and lints the diff for style "
        "violations, failing the build before a human reviewer even looks at it if any of "
        "those checks do not pass. Only after the pipeline is green and a colleague has "
        "approved the change does it get merged into the main branch and queued for the "
        "next deployment."
    ],
    "math": [
        "Algebra gives us a systematic way to find unknown quantities from known "
        "relationships. To solve the equation 3x plus 7 equals 22, we isolate x by "
        "subtracting 7 from both sides to get 3x equals 15, then dividing both sides by 3 to "
        "find that x equals 5. This same principle of performing identical operations on "
        "both sides of an equation scales up to systems of many variables, which is the "
        "foundation of linear algebra and, eventually, of the matrix operations that power "
        "modern machine learning models.\n\n"
        "Calculus formalizes the idea of instantaneous change. The derivative of a function "
        "at a point measures the slope of the tangent line there, so the derivative of x "
        "cubed plus 2x with respect to x is 3x squared plus 2, telling us exactly how fast "
        "the function's output grows as x increases. Integration reverses this process, "
        "accumulating infinitely many infinitesimal slices to compute a total, such as the "
        "area under a curve or the distance traveled by an object whose velocity changes "
        "continuously over time.\n\n"
        "Geometry and number theory each contribute their own foundational facts. The "
        "Pythagorean theorem states that in a right triangle, the square of the hypotenuse "
        "equals the sum of the squares of the other two sides, a relationship that underlies "
        "everything from architectural design to GPS trilateration. A prime number, "
        "meanwhile, is a whole number greater than one that is divisible only by itself and "
        "one; the fact that every integer factors uniquely into primes is the basis of much "
        "of modern cryptography.\n\n"
        "Probability quantifies uncertainty using precise rules rather than intuition alone. "
        "If a fair six-sided die is rolled twice, the chance of rolling a six both times is "
        "one-sixth multiplied by one-sixth, or one in thirty-six, because the two rolls are "
        "independent events. This same multiplication rule, extended across thousands of "
        "variables, is what allows statisticians to model everything from election outcomes "
        "to the reliability of a manufactured part over its expected lifetime."
    ],
    "biomedical": [
        "A 58-year-old woman arrived at the emergency department reporting sudden, crushing "
        "chest pain that radiated into her jaw, along with nausea and cold sweats. An "
        "electrocardiogram showed ST-segment elevation in the anterior leads, consistent "
        "with an acute myocardial infarction, and she was taken directly to the "
        "catheterization lab, where an interventional cardiologist located and cleared a "
        "blockage in the left anterior descending artery. Within an hour of the blocked "
        "vessel being reopened, her chest pain had resolved and her cardiac enzyme levels "
        "began trending back toward normal.\n\n"
        "In a separate randomized, double-blind trial, researchers compared a new "
        "anti-inflammatory therapy against a placebo in patients with a chronic autoimmune "
        "condition. Participants who received the active treatment showed a statistically "
        "significant reduction in joint swelling and reported less pain on standardized "
        "questionnaires after twelve weeks, though a minority experienced mild "
        "injection-site irritation. The investigators concluded that the therapy was both "
        "effective and well tolerated, though they recommended a larger, multi-site "
        "follow-up trial before it could be considered for regulatory approval.\n\n"
        "At the molecular level, many of these therapies work by binding to a specific "
        "cell-surface receptor and blocking a signaling cascade that would otherwise trigger "
        "inflammation. This interrupts the release of pro-inflammatory cytokines, small "
        "proteins that normally recruit additional immune cells to a site of injury or "
        "infection, dampening the immune response without shutting it down entirely. A "
        "tissue biopsy taken before and after treatment can confirm this mechanism directly, "
        "typically showing reduced immune cell infiltration and lower levels of inflammatory "
        "markers such as C-reactive protein in the blood.\n\n"
        "Preventive medicine remains one of the most cost-effective tools available to "
        "clinicians. Routine vaccination trains the immune system to recognize a pathogen's "
        "distinctive surface proteins well before a real infection occurs, so that "
        "antibodies and memory immune cells are already circulating by the time exposure "
        "happens. Regular screening, similarly, catches conditions like hypertension or "
        "early-stage cancer while they are still asymptomatic and far easier to treat, often "
        "years before they would otherwise have produced any noticeable symptoms."
    ],
    "legal": [
        "This Master Services Agreement is entered into between the Client and the Service "
        "Provider as of the Effective Date, and governs all statements of work executed "
        "under it. The Service Provider agrees to deliver the services described in each "
        "statement of work in a professional and workmanlike manner, and the Client agrees "
        "to pay all undisputed invoices within thirty days of receipt. Either party may "
        "terminate the Agreement for convenience upon sixty days' written notice, provided "
        "that any fees accrued for work performed prior to the termination date remain due "
        "and payable in full.\n\n"
        "In a subsequent dispute, the plaintiff alleged that the defendant had breached a "
        "supply agreement by failing to deliver conforming goods by the contractually "
        "specified deadline. At trial, the court heard testimony from an industry expert "
        "regarding customary delivery timelines, together with internal correspondence in "
        "which the defendant acknowledged awareness of the deadline and its likely inability "
        "to meet it. The jury found that the defendant's failure to perform was a material "
        "breach and awarded damages calculated to place the plaintiff in the position it "
        "would have occupied had the contract been properly performed.\n\n"
        "On appeal, the defendant argued that the trial court's jury instructions on "
        "materiality were legally deficient and warranted a new trial. The appellate panel "
        "disagreed, holding that the instructions, considered in their entirety, correctly "
        "stated the governing legal standard, and that any imprecision in a single sentence "
        "did not amount to reversible error given the overwhelming weight of the evidence "
        "presented. The panel further reaffirmed that in civil actions the burden rests on "
        "the plaintiff to establish each element of the claim by a preponderance of the "
        "evidence, a standard it found comfortably satisfied on this record.\n\n"
        "Beyond contract and tort claims, corporate counsel also spend considerable time on "
        "regulatory compliance. Before launching a new product in a foreign jurisdiction, a "
        "company typically commissions a legal opinion addressing local licensing "
        "requirements, data protection obligations, and any sector-specific restrictions that "
        "might apply, since noncompliance can expose the company to fines, injunctions, or "
        "the forced withdrawal of the product from that market entirely."
    ],
    "creative_writing": [
        "The lighthouse keeper had watched a thousand storms roll in off the grey Atlantic, "
        "but something about this one made him pause at the window with his tea going cold "
        "in his hand. The waves were climbing higher than the rocks that had stood against "
        "them for three hundred years, and for the first time in his forty seasons on the "
        "island, he found himself counting the ships he could see and hoping the count would "
        "not change by morning.\n\n"
        "Mira found the letter tucked inside a hollowed-out book on her grandmother's shelf, "
        "the paper gone soft and yellow at the folds. Her hands trembled as she unfolded it, "
        "not from cold but from the particular fear of learning something that could not be "
        "unlearned, and when she finally read the first line, she understood at once why it "
        "had been hidden rather than simply thrown away.\n\n"
        "Deep in the forest, where the canopy grew so thick that noon light arrived the color "
        "of dusk, the old paths remembered every traveler who had ever walked them. The wind "
        "moved through the high branches in long, unhurried sighs, and if you stood still "
        "long enough and let your own breathing slow to match it, you could almost believe "
        "the trees were arguing quietly among themselves about whether to let you pass.\n\n"
        "By the time the last streetlamp flickered out, the city had already begun its other "
        "life, the one that belonged to the people who swept its floors and stocked its "
        "shelves while everyone else slept. A fox slipped across the empty intersection "
        "without breaking stride, entirely unbothered by the traffic lights still cycling to "
        "no one, and somewhere above the rooftops the sky was already deciding, slowly, what "
        "color the morning would be."
    ],
    "conversational": [
        "Hey, sorry for the late reply, my phone died on the way home and I didn't get a "
        "chance to charge it until just now. Anyway, are we still good for Saturday, or did "
        "something come up on your end? I can also do Sunday afternoon if that works better, "
        "just let me know so I can figure out the rest of my weekend around it.\n\n"
        "Honestly, I've been kind of exhausted this week, nothing serious, just one of those "
        "stretches where every day feels a little longer than it should. I think I just need "
        "a weekend where I don't have anywhere to be, maybe cook something simple, watch a "
        "movie I've already seen a dozen times, that kind of thing. How about you, anything "
        "fun happen lately, or has it been the same kind of week over there?\n\n"
        "Oh, that reminds me, did you end up trying that new place downtown? A couple of "
        "people at work were talking about it and apparently the line gets pretty long on "
        "weekends, so if we want to check it out we should probably go early or just do a "
        "weekday evening instead. I'm not picky either way, honestly whatever's easiest works "
        "for me, I just haven't had a good excuse to get out of the house in a while.\n\n"
        "Thanks again for helping me move that bookshelf last week, by the way, I really owe "
        "you one. Let me know if you ever need a hand with anything, moving, fixing something "
        "around the house, whatever, I'm around most weekends these days. Talk soon, and "
        "text me whenever about Saturday, no rush."
    ],
}

domains = list(DOMAIN_PROMPTS.keys())
print(f"Domains: {domains}")
for domain, prompts in DOMAIN_PROMPTS.items():
    assert len(prompts) == 1, f"{domain} has {len(prompts)} prompts, expected 1"
    print(f"  {domain}: {len(prompts[0].split())} words")


In [ ]:
def stats_for_prompt(prompt):
    """Returns per-layer [num_experts] top-k hit counts and summed probs, plus token count,
    plus each token's real decoded text and, per layer/expert, which (token index, routing
    score) pairs actually selected that expert in their real top-k -- the score lets callers
    rank tokens by how strongly they activated the expert, not just occurrence order."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs, output_router_logits=True)
    router_logits = outputs.router_logits  # tuple of [seq, num_experts], one per layer
    token_ids = inputs["input_ids"][0].tolist()
    token_strs = [tokenizer.decode([tid]) for tid in token_ids]
    n_tokens = len(token_ids)

    hit_counts = torch.zeros(num_layers, num_experts)
    prob_sums = torch.zeros(num_layers, num_experts)
    expert_token_idx = [[[] for _ in range(num_experts)] for _ in range(num_layers)]
    for li, layer_logits in enumerate(router_logits):
        probs = torch.softmax(layer_logits.float(), dim=-1).cpu()  # [seq, num_experts]
        topk = torch.topk(probs, k=top_k_experts, dim=-1).indices  # [seq, top_k]
        for t in range(n_tokens):
            hit_counts[li, topk[t]] += 1
            for e in topk[t].tolist():
                expert_token_idx[li][e].append((t, float(probs[t, e])))
        prob_sums[li] += probs.sum(dim=0)
    return hit_counts, prob_sums, n_tokens, token_strs, expert_token_idx


In [ ]:
activation_rate = {}   # domain -> [layer][expert]  fraction of domain's tokens with expert in top-k
avg_prob = {}           # domain -> [layer][expert]  mean router softmax prob (selected or not)
token_counts = {}
prompt_counts = {}
# domain -> [token_str, ...] and domain -> [layer][expert] -> [token_idx, ...] into that
# list, so the popup can show exactly which real tokens routed to a given expert/layer.
domain_tokens = {}
expert_token_idx = {}

for domain, prompts in DOMAIN_PROMPTS.items():
    print(f"\n== domain: {domain} ==")
    assert len(prompts) == 1, "domain_tokens/expert_token_idx assume exactly one prompt per domain"
    prompt = prompts[0]
    print(f"  {prompt[:80]!r}...")
    hits, probs, n_tok, token_strs, e_idx = stats_for_prompt(prompt)
    activation_rate[domain] = (hits / n_tok).tolist()
    avg_prob[domain] = (probs / n_tok).tolist()
    token_counts[domain] = n_tok
    prompt_counts[domain] = len(prompts)
    domain_tokens[domain] = token_strs
    expert_token_idx[domain] = e_idx
    print(f"  total tokens: {n_tok}")


In [ ]:
# Synthetic baseline: none of the 6 domains is meant to be neutral/generic text, so instead
# of a 7th hand-authored "baseline" passage, use the mean activation rate across the 6
# domains themselves, per (layer, expert), as the reference point for specialization_score
# and layer_divergence.
EPS = 1e-4
baseline_rate = [
    [sum(activation_rate[d][li][e] for d in domains) / len(domains) for e in range(num_experts)]
    for li in range(num_layers)
]

# specialization_score[domain][layer][expert] = log2((rate_domain + eps) / (rate_baseline + eps))
# vs the synthetic baseline -- positive = over-used relative to the 6-domain average.
specialization_score = {}
for domain in domains:
    rate = activation_rate[domain]
    specialization_score[domain] = [
        [
            round(
                torch.log2(torch.tensor((rate[li][e] + EPS) / (baseline_rate[li][e] + EPS))).item(),
                4,
            )
            for e in range(num_experts)
        ]
        for li in range(num_layers)
    ]

# layer_divergence[domain][layer] = total-variation distance between domain's and the
# synthetic baseline's per-expert selection distribution (each normalized to sum to 1 across
# experts via /top_k). 0 = identical routing to the 6-domain average at that layer, 1 =
# completely disjoint expert sets. Kept as supplementary context (shown in the click popup);
# the primary chart plots domain_rate directly for all 6 domains.
layer_divergence = {}
for domain in domains:
    divs = []
    for li in range(num_layers):
        dom_dist = [activation_rate[domain][li][e] / top_k_experts for e in range(num_experts)]
        base_dist = [baseline_rate[li][e] / top_k_experts for e in range(num_experts)]
        tv = 0.5 * sum(abs(a - b) for a, b in zip(dom_dist, base_dist))
        divs.append(round(tv, 5))
    layer_divergence[domain] = divs

# domain_rate[domain][layer] = mean activation_rate of that domain's top-K (=top_k_experts)
# most-used experts at that layer -- a single self-contained number per domain per layer,
# computed identically across all 6 domains so they can be plotted side by side.
domain_rate = {}
for domain in domains:
    rates = []
    for li in range(num_layers):
        top_vals = sorted(activation_rate[domain][li], reverse=True)[:top_k_experts]
        rates.append(round(sum(top_vals) / len(top_vals), 5))
    domain_rate[domain] = rates

# top experts per domain, ranked directly by real activation_rate (no baseline comparison)
top_specialists = {}
for domain in domains:
    pairs = []
    for li in range(num_layers):
        for e in range(num_experts):
            pairs.append((activation_rate[domain][li][e], li, e))
    pairs.sort(reverse=True)
    top_specialists[domain] = [
        {"layer": li, "expert": e, "activation_rate": round(rate, 4)}
        for rate, li, e in pairs[:12]
    ]


In [ ]:
out = {
    "domains": domains,
    "num_layers": num_layers,
    "num_experts": num_experts,
    "top_k_experts": top_k_experts,
    "token_counts": token_counts,
    "prompt_counts": prompt_counts,
    "example_prompts": DOMAIN_PROMPTS,
    "activation_rate": {d: [[round(v, 5) for v in row] for row in activation_rate[d]] for d in domains},
    "avg_prob": {d: [[round(v, 5) for v in row] for row in avg_prob[d]] for d in domains},
    "specialization_score": specialization_score,
    "layer_divergence": layer_divergence,
    "domain_rate": domain_rate,
    "domain_tokens": domain_tokens,
    "expert_token_idx": expert_token_idx,
    "top_specialists": top_specialists,
}

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(out, f)

print(f"\nWrote domain specialization data to {OUT_PATH} ({os.path.getsize(OUT_PATH) / 1e3:.1f} KB)")


## UMAP: (layer, expert) activation across domains

Reuses the `activation_rate` computed above (no extra forward passes) to build one vector
per (layer, expert) pair, one dimension per domain, and projects it to 2D with cosine-metric
UMAP -- the same method used in `extract_routing_trace.ipynb`. All-zero (never-activated)
pairs are excluded from the projection and reported separately as `excluded_experts`.


In [ ]:
import umap

NUM_LAYERS = num_layers
NUM_EXPERTS = num_experts

expert_vectors = np.array([
    [activation_rate[d][li][e] for d in domains]
    for li in range(NUM_LAYERS)
    for e in range(NUM_EXPERTS)
])

point_layer_ids = np.repeat(np.arange(NUM_LAYERS), NUM_EXPERTS)
point_expert_ids = np.tile(np.arange(NUM_EXPERTS), NUM_LAYERS)

# Never-activated (layer, expert) pairs are all-zero and undefined under the cosine metric --
# exclude from the projection, report separately as excluded_experts.
active_mask = expert_vectors.sum(axis=1) > 0
active_vectors = expert_vectors[active_mask]

reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1, metric="cosine", n_jobs=1)
active_embedding = reducer.fit_transform(active_vectors)

assert not np.isnan(active_embedding).any(), (
    "UMAP produced NaN coordinates even after excluding all-zero rows -- inspect "
    "active_vectors for degenerate rows, or re-run with metric='euclidean'."
)

print(f"Built {expert_vectors.shape[0]} (layer, expert) vectors across {len(domains)} domains.")
print(f"UMAP embedding shape: {active_embedding.shape} ({int(active_mask.sum())} active of {expert_vectors.shape[0]} total pairs)")

TOP_K_TOKENS = 8
active_indices = np.flatnonzero(active_mask)

umap_points = []
for row, i in enumerate(active_indices):
    layer_id = int(point_layer_ids[i])
    expert_id = int(point_expert_ids[i])
    vec = expert_vectors[i]
    dominant_domain = domains[int(np.argmax(vec))]

    # top_tokens: pool every (token, routing score) pair that selected this (layer, expert)
    # across all domains, then keep the TOP_K_TOKENS with the highest score -- matching
    # extract_routing.ipynb's ranked-by-score approach, so the hover popup surfaces the
    # tokens that activated this expert most strongly, not just the first ones encountered.
    pooled = [
        (score, domain_tokens[d][t_idx], d)
        for d in domains
        for t_idx, score in expert_token_idx[d][layer_id][expert_id]
    ]
    pooled.sort(key=lambda item: -item[0])
    top_tokens = [
        {"token": tok, "score": round(float(score), 4), "domain": d}
        for score, tok, d in pooled[:TOP_K_TOKENS]
    ]

    umap_points.append({
        "layer_id": layer_id,
        "expert_id": expert_id,
        "x": round(float(active_embedding[row, 0]), 4),
        "y": round(float(active_embedding[row, 1]), 4),
        "dominant_domain": dominant_domain,
        "domain_activation_rate": {d: round(float(vec[j]), 4) for j, d in enumerate(domains)},
        "top_tokens": top_tokens,
    })

excluded_experts = [
    {"layer_id": int(point_layer_ids[i]), "expert_id": int(point_expert_ids[i])}
    for i in np.flatnonzero(~active_mask)
]

assert len(umap_points) + len(excluded_experts) == NUM_LAYERS * NUM_EXPERTS

umap_data = {
    "domains": domains,
    "num_layers": NUM_LAYERS,
    "num_experts": NUM_EXPERTS,
    "points": umap_points,
    "excluded_experts": excluded_experts,
}

with open(UMAP_OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(umap_data, f, ensure_ascii=False, allow_nan=False, indent=2)

print(f"Wrote {UMAP_OUT_PATH} ({len(umap_points)} points, {len(excluded_experts)} excluded pairs)")


In [ ]:
try:
    from google.colab import files
    files.download(OUT_PATH)
    files.download(UMAP_OUT_PATH)
except ImportError:
    print("Not running in Google Colab -- skipping auto-download.")
    print(f"Files were written locally at: {OUT_PATH} and {UMAP_OUT_PATH}")
